# Cost Optimization Strategies

This notebook turns the three cost optimization examples from the README into interactive cells:

- Prompt optimization reduces unnecessary input tokens before a request is sent.
- Semantic caching avoids repeated LLM calls for similar questions.
- Smart model routing sends each task to the cheapest model that should handle it.

Live LLM calls are disabled by default with `RUN_LIVE_API_CALLS = False` so the notebook can be executed safely while you inspect the code.

## Shared Setup

The examples use Langfuse spans and generations so cost-saving decisions show up in the trace metadata.

In [2]:
import hashlib
import json
import re
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Optional

from dotenv import load_dotenv
from langfuse import get_client, observe

load_dotenv()

langfuse = get_client()
RUN_LIVE_API_CALLS = True

## Prompt Optimization

This helper removes common filler phrases, collapses whitespace, removes repeated sentence-level instructions, and records before/after token estimates in Langfuse. It is useful for mechanical cleanup, but manual review is still needed to preserve prompt quality.

In [3]:
# Common phrases that usually add tokens without adding useful instruction.
FILLER_PHRASES = [
    "I want you to",
    "You are a highly intelligent",
    "Please note that",
    "It's important to remember that",
    "In your response, make sure to",
    "As an AI assistant,",
]


def estimate_tokens(text: str) -> int:
    """Estimate token count without adding tokenizer dependencies."""
    return max(1, round(len(text) / 4)) if text else 0


@observe(name="optimize_prompt", as_type="span")
def optimize_prompt(prompt: str) -> str:
    """Remove obvious prompt bloat while preserving unique instructions."""
    original_prompt = prompt

    # Remove common filler phrases with case-insensitive matching.
    optimized = prompt
    for filler in FILLER_PHRASES:
        optimized = re.sub(re.escape(filler), "", optimized, flags=re.IGNORECASE)

    # Collapse repeated whitespace introduced by removals.
    optimized = " ".join(optimized.split())

    # Remove repeated sentence-level instructions while preserving order.
    sentences = re.split(r"(?<=[.!?])\s+", optimized)
    seen: set[str] = set()
    unique_sentences: list[str] = []
    for sentence in sentences:
        normalized = sentence.strip().lower()
        if normalized and normalized not in seen:
            seen.add(normalized)
            unique_sentences.append(sentence.strip())

    optimized = " ".join(unique_sentences)

    original_tokens = estimate_tokens(original_prompt)
    optimized_tokens = estimate_tokens(optimized)
    reduction_pct = (
        ((original_tokens - optimized_tokens) / original_tokens) * 100
        if original_tokens
        else 0.0
    )

    # Record the optimization metrics in the current Langfuse span.
    langfuse.update_current_span(
        input={"prompt": original_prompt},
        output={"optimized_prompt": optimized},
        metadata={
            "original_chars": len(original_prompt),
            "optimized_chars": len(optimized),
            "estimated_original_tokens": original_tokens,
            "estimated_optimized_tokens": optimized_tokens,
            "estimated_token_reduction_pct": round(reduction_pct, 2),
            "fillers_checked": FILLER_PHRASES,
        },
    )

    return optimized

In [4]:
bloated_prompt = """
As an AI assistant, I want you to explain observability.
Please note that it's important to remember that in your response, make sure to be accurate.
In your response, make sure to be accurate.
Be concise.
"""

optimized_prompt = optimize_prompt(bloated_prompt)

print("Original prompt:")
print(bloated_prompt.strip())
print("\nOptimized prompt:")
print(optimized_prompt)
print("\nEstimated token reduction:")
print(estimate_tokens(bloated_prompt), "->", estimate_tokens(optimized_prompt))

Original prompt:
As an AI assistant, I want you to explain observability.
Please note that it's important to remember that in your response, make sure to be accurate.
In your response, make sure to be accurate.
Be concise.

Optimized prompt:
explain observability. be accurate. Be concise.

Estimated token reduction:
52 -> 12


## Semantic Caching

Semantic caching stores query embeddings and responses in ChromaDB. A new query is embedded, compared against previous queries, and reused when the semantic similarity is high enough and the cache entry is not expired.

In [5]:
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer

BASE_DIR = Path.cwd()
CACHE_DIR = BASE_DIR / "chroma_cache_db"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
GENERATION_MODEL = "claude-sonnet-4-6"

anthropic_client = anthropic.Anthropic()


@dataclass
class CacheLookup:
    """Result returned by the semantic cache lookup."""

    response: str
    similarity: float
    cached_query: str
    age_seconds: float


@observe(name="call_claude_for_cache_miss", as_type="generation")
def call_claude_for_cache_miss(query: str) -> str:
    """Call Claude only when the semantic cache misses."""
    response = anthropic_client.messages.create(
        model=GENERATION_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": query}],
    )
    response_text = response.content[0].text

    langfuse.update_current_generation(
        model=GENERATION_MODEL,
        input=[{"role": "user", "content": query}],
        output=response_text,
        usage_details={
            "input": response.usage.input_tokens,
            "output": response.usage.output_tokens,
            "total": response.usage.input_tokens + response.usage.output_tokens,
        },
        metadata={"cache_hit": False},
    )
    return response_text


class SemanticCache:
    """Persistent ChromaDB-backed semantic response cache."""

    def __init__(
        self,
        similarity_threshold: float = 0.92,
        ttl_hours: int = 24,
        persist_directory: str | Path = CACHE_DIR,
    ) -> None:
        self.client = chromadb.PersistentClient(path=str(persist_directory))
        self.collection = self.client.get_or_create_collection(
            name="llm_cache",
            metadata={"hnsw:space": "cosine"},
        )
        self.encoder = SentenceTransformer(EMBEDDING_MODEL_NAME)
        self.threshold = similarity_threshold
        self.ttl = timedelta(hours=ttl_hours)

    def _get_embedding(self, text: str) -> list[float]:
        return self.encoder.encode(text).tolist()

    def _is_expired(self, timestamp: str) -> tuple[bool, float]:
        cached_time = datetime.fromisoformat(timestamp)
        if cached_time.tzinfo is None:
            cached_time = cached_time.replace(tzinfo=timezone.utc)

        age = datetime.now(timezone.utc) - cached_time
        return age > self.ttl, age.total_seconds()

    @observe(name="semantic_cache_get", as_type="retriever")
    def get(self, query: str) -> CacheLookup | None:
        """Return a cached response when a semantically similar query exists."""
        query_embedding = self._get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=1,
            include=["documents", "metadatas", "distances"],
        )

        documents = results.get("documents") or [[]]
        metadatas = results.get("metadatas") or [[]]
        distances = results.get("distances") or [[]]

        if not documents[0]:
            langfuse.update_current_span(
                output={"cache_hit": False},
                metadata={"reason": "empty_cache", "threshold": self.threshold},
            )
            return None

        distance = distances[0][0]
        similarity = 1 - distance
        metadata = metadatas[0][0]
        expired, age_seconds = self._is_expired(metadata["timestamp"])

        if similarity < self.threshold:
            langfuse.update_current_span(
                output={"cache_hit": False},
                metadata={
                    "reason": "below_threshold",
                    "similarity": similarity,
                    "threshold": self.threshold,
                    "cached_query": documents[0][0],
                },
            )
            return None

        if expired:
            langfuse.update_current_span(
                output={"cache_hit": False},
                metadata={
                    "reason": "expired",
                    "similarity": similarity,
                    "ttl_hours": self.ttl.total_seconds() / 3600,
                    "age_seconds": age_seconds,
                    "cached_query": documents[0][0],
                },
            )
            return None

        cached_response = json.loads(metadata["response"])
        lookup = CacheLookup(
            response=cached_response,
            similarity=similarity,
            cached_query=documents[0][0],
            age_seconds=age_seconds,
        )

        langfuse.update_current_span(
            output={
                "cache_hit": True,
                "cached_query": lookup.cached_query,
                "response_preview": lookup.response[:240],
            },
            metadata={
                "similarity": lookup.similarity,
                "threshold": self.threshold,
                "age_seconds": lookup.age_seconds,
            },
        )
        return lookup

    @observe(name="semantic_cache_set", as_type="embedding")
    def set(
        self,
        query: str,
        response: str,
        metadata: dict[str, Any] | None = None,
    ) -> None:
        """Store a query embedding and response for future semantic matches."""
        query_embedding = self._get_embedding(query)
        doc_id = hashlib.sha256(query.encode("utf-8")).hexdigest()
        timestamp = datetime.now(timezone.utc).isoformat()

        self.collection.upsert(
            ids=[doc_id],
            embeddings=[query_embedding],
            documents=[query],
            metadatas=[
                {
                    "response": json.dumps(response),
                    "timestamp": timestamp,
                    **(metadata or {}),
                }
            ],
        )

        langfuse.update_current_span(
            output={"cached": True, "doc_id": doc_id},
            metadata={
                "query_length": len(query),
                "embedding_model": EMBEDDING_MODEL_NAME,
                "embedding_dim": len(query_embedding),
                "timestamp": timestamp,
            },
        )

In [6]:
@observe(name="cached_llm_call", as_type="span")
def cached_llm_call(query: str, cache: SemanticCache) -> str:
    """Check semantic cache before calling the LLM."""
    cached = cache.get(query)
    if cached:
        langfuse.update_current_span(
            output={"response": cached.response},
            metadata={
                "cache_hit": True,
                "similarity": cached.similarity,
                "cached_query": cached.cached_query,
                "api_call_saved": True,
            },
        )
        print(f"  CACHE HIT - similarity: {cached.similarity:.2%}")
        return cached.response

    print("  CACHE MISS - calling Claude API")
    response = call_claude_for_cache_miss(query)
    cache.set(query, response, metadata={"source": "claude"})

    langfuse.update_current_span(
        output={"response": response},
        metadata={"cache_hit": False, "api_call_saved": False},
    )
    return response


@observe(name="simulate_semantic_cache", as_type="span")
def simulate_semantic_cache(cache: SemanticCache) -> dict[str, float | int]:
    """Demonstrate semantic cache hits with similar questions."""
    questions = [
        "What is a Python list comprehension?",
        "Explain list comprehensions in Python",
        "What is the difference between supervised and unsupervised learning?",
        "Explain supervised vs unsupervised machine learning",
    ]

    total_queries = 0
    cache_hits = 0
    cache_misses = 0

    for question in questions:
        total_queries += 1
        print(f'Query {total_queries}: "{question}"')

        before = cache.get(question)
        if before:
            cache_hits += 1
        else:
            cache_misses += 1

        response = cached_llm_call(question, cache)
        display_response = response[:120] + "..." if len(response) > 120 else response
        print(f"  Response: {display_response}")

    hit_rate = (cache_hits / total_queries * 100) if total_queries else 0.0
    summary = {
        "total_queries": total_queries,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "hit_rate_pct": round(hit_rate, 2),
        "api_calls_saved": cache_hits,
    }

    langfuse.update_current_span(output=summary, metadata=summary)
    return summary


if RUN_LIVE_API_CALLS:
    cache = SemanticCache(similarity_threshold=0.85)
    simulate_semantic_cache(cache)
else:
    print("Semantic cache live demo skipped. Set RUN_LIVE_API_CALLS = True to run it.")

Query 1: "What is a Python list comprehension?"
  CACHE MISS - calling Claude API
  Response: ## Python List Comprehension

A list comprehension is a concise, readable way to create a new list by applying an expres...
Query 2: "Explain list comprehensions in Python"
  CACHE HIT - similarity: 91.47%
  Response: ## Python List Comprehension

A list comprehension is a concise, readable way to create a new list by applying an expres...
Query 3: "What is the difference between supervised and unsupervised learning?"
  CACHE MISS - calling Claude API
  Response: # Supervised vs. Unsupervised Learning

## Supervised Learning
**Training data includes labeled examples** (input + corr...
Query 4: "Explain supervised vs unsupervised machine learning"
  CACHE MISS - calling Claude API
  Response: # Supervised vs Unsupervised Machine Learning

## Supervised Learning

**The model learns from labeled training data** —...


## Smart Model Routing

The router classifies each prompt into a task type, maps that task to a cost-appropriate model, and records the routing decision in Langfuse. The first demo cell below only classifies prompts, so it does not call any provider.

In [7]:
from anthropic import Anthropic
from openai import OpenAI


class TaskType(Enum):
    SIMPLE = "simple"  # Yes/no answers and basic classification.
    MODERATE = "moderate"  # Summarization and information extraction.
    COMPLEX = "complex"  # Analysis, comparison, and reasoning.
    CODE = "code"  # Code generation or debugging.
    CREATIVE = "creative"  # Creative writing tasks.


@dataclass
class ModelCallResult:
    """Normalized response data across Anthropic and OpenAI calls."""

    content: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated_cost: float
    duration_ms: float


PRICING = {
    # Prices are USD per 1M tokens; keep this table aligned with vendor pricing pages.
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
}


class ModelRouter:
    """Route requests to the lowest-cost model that should handle the task."""

    MODELS = {
        TaskType.SIMPLE: "claude-haiku-4-5-20251001",
        TaskType.MODERATE: "gpt-4o-mini",
        TaskType.CODE: "claude-sonnet-4-6",
        TaskType.COMPLEX: "claude-sonnet-4-6",
        TaskType.CREATIVE: "gpt-4o",
    }

    def classify_task(self, prompt: str) -> TaskType:
        """Classify a prompt with simple, explainable keyword patterns."""
        prompt_lower = prompt.lower()

        simple_patterns = [
            r"\b(yes or no)\b",
            r"\b(true or false)\b",
            r"\b(classify|categorize)\b",
            r"^is (this|it|the)",
            r"\b(which one|choose|select)\b",
        ]
        if any(re.search(pattern, prompt_lower) for pattern in simple_patterns):
            return TaskType.SIMPLE

        code_patterns = [
            r"\b(write|create|generate|fix|debug).*(code|function|class|script)\b",
            r"\b(python|javascript|typescript|java|rust)\b",
            r"```",
        ]
        if any(re.search(pattern, prompt_lower) for pattern in code_patterns):
            return TaskType.CODE

        complex_patterns = [
            r"\b(analyze|evaluate|compare|critique)\b",
            r"\b(why|how).*(work|happen|cause)\b",
            r"\b(pros and cons|trade-?offs)\b",
            r"\b(explain.*(detail|depth))\b",
        ]
        if any(re.search(pattern, prompt_lower) for pattern in complex_patterns):
            return TaskType.COMPLEX

        creative_patterns = [
            r"\b(write|create|compose).*(story|poem|essay|blog)\b",
            r"\b(creative|imaginative|original)\b",
        ]
        if any(re.search(pattern, prompt_lower) for pattern in creative_patterns):
            return TaskType.CREATIVE

        return TaskType.MODERATE

    def route(self, prompt: str, override_model: Optional[str] = None) -> str:
        """Return an explicit override or the model mapped to the detected task."""
        if override_model:
            return override_model

        task_type = self.classify_task(prompt)
        return self.MODELS[task_type]


router = ModelRouter()
anthropic_client = Anthropic()
openai_client = OpenAI()


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Estimate provider cost from the local pricing table."""
    pricing = PRICING.get(model, {"input": 0.0, "output": 0.0})
    return (
        input_tokens * pricing["input"] / 1_000_000
        + output_tokens * pricing["output"] / 1_000_000
    )

In [8]:
example_prompts = [
    "Is 2 + 2 = 4? Yes or no",
    "Write a Python function to sort a list",
    "Compare prompt caching and semantic caching in detail",
    "Write a short blog post about observability",
]

for prompt in example_prompts:
    task_type = router.classify_task(prompt)
    selected_model = router.route(prompt)
    print(f"{task_type.value:>8} -> {selected_model}: {prompt}")

  simple -> claude-haiku-4-5-20251001: Is 2 + 2 = 4? Yes or no
    code -> claude-sonnet-4-6: Write a Python function to sort a list
 complex -> claude-sonnet-4-6: Compare prompt caching and semantic caching in detail
creative -> gpt-4o: Write a short blog post about observability


In [9]:
@observe(name="call_claude_routed", as_type="generation")
def call_claude_routed(prompt: str, model: str, max_tokens: int = 1024) -> ModelCallResult:
    """Call Anthropic and record the provider request as a Langfuse generation."""
    start = time.perf_counter()
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )

    content = response.content[0].text
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    total_tokens = input_tokens + output_tokens
    cost = estimate_cost(model, input_tokens, output_tokens)
    duration_ms = (time.perf_counter() - start) * 1000

    langfuse.update_current_generation(
        model=model,
        input=[{"role": "user", "content": prompt}],
        output=content,
        usage_details={"input": input_tokens, "output": output_tokens, "total": total_tokens},
        cost_details={"total": cost},
        metadata={"provider": "anthropic", "duration_ms": duration_ms},
    )

    return ModelCallResult(content, "anthropic", model, input_tokens, output_tokens, total_tokens, cost, duration_ms)


@observe(name="call_openai_routed", as_type="generation")
def call_openai_routed(prompt: str, model: str) -> ModelCallResult:
    """Call OpenAI and record the provider request as a Langfuse generation."""
    start = time.perf_counter()
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    content = response.choices[0].message.content or ""
    input_tokens = response.usage.prompt_tokens if response.usage else 0
    output_tokens = response.usage.completion_tokens if response.usage else 0
    total_tokens = response.usage.total_tokens if response.usage else 0
    cost = estimate_cost(model, input_tokens, output_tokens)
    duration_ms = (time.perf_counter() - start) * 1000

    langfuse.update_current_generation(
        model=model,
        input=[{"role": "user", "content": prompt}],
        output=content,
        usage_details={"input": input_tokens, "output": output_tokens, "total": total_tokens},
        cost_details={"total": cost},
        metadata={"provider": "openai", "duration_ms": duration_ms},
    )

    return ModelCallResult(content, "openai", model, input_tokens, output_tokens, total_tokens, cost, duration_ms)


@observe(name="routed_llm_call", as_type="span")
def routed_llm_call(prompt: str, override_model: Optional[str] = None) -> str:
    """Classify the prompt, route it, call the provider, and trace the decision."""
    task_type = router.classify_task(prompt)
    selected_model = router.route(prompt, override_model)

    if selected_model.startswith("claude"):
        result = call_claude_routed(prompt, model=selected_model)
    else:
        result = call_openai_routed(prompt, model=selected_model)

    langfuse.update_current_span(
        input={"prompt": prompt, "override_model": override_model},
        output={"content": result.content},
        metadata={
            "task_type": task_type.value,
            "routed_model": selected_model,
            "override_used": override_model is not None,
            "provider": result.provider,
            "input_tokens": result.input_tokens,
            "output_tokens": result.output_tokens,
            "total_tokens": result.total_tokens,
            "estimated_cost": result.estimated_cost,
            "duration_ms": result.duration_ms,
        },
    )

    return result.content


if RUN_LIVE_API_CALLS:
    print(routed_llm_call("Is 2 + 2 = 4? Yes or no"))
    print(routed_llm_call("Write a Python function to sort a list"))
    langfuse.flush()
else:
    print("Model routing live demo skipped. Set RUN_LIVE_API_CALLS = True to run it.")

Yes.
## Sorting a List in Python

Here are several ways to sort a list in Python:

### Basic Sort Function

```python
def sort_list(lst, reverse=False):
    """
    Sort a list in ascending or descending order.
    
    Args:
        lst: The list to sort
        reverse: If True, sort in descending order (default: False)
    
    Returns:
        A new sorted list
    """
    return sorted(lst, reverse=reverse)
```

### Examples of Usage

```python
# Sort numbers
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print(sort_list(numbers))           # [1, 1, 2, 3, 4, 5, 6, 9]
print(sort_list(numbers, reverse=True))  # [9, 6, 5, 4, 3, 2, 1, 1]

# Sort strings
words = ["banana", "apple", "cherry", "date"]
print(sort_list(words))             # ['apple', 'banana', 'cherry', 'date']

# Sort strings by length
def sort_by_length(lst):
    return sorted(lst, key=len)

print(sort_by_length(words))        # ['date', 'apple', 'banana', 'cherry']

# Sort list of dictionaries
people = [
    {"name": "Alice", "age"